# Pauli Algebra 06: Lie Closure and Heisenberg Dynamics

This notebook follows the Lie-closure workflow described in `quantum_simulation/pauli_algebra/README.md` and `evolution_problem_lie_algebra_notes.md`.


## What you will learn

1. How to generate a symbolic Lie-closure basis from a Hamiltonian and observable.
2. How to project operators into that basis.
3. How to evolve an observable in the Heisenberg picture through the adjoint matrix.
4. How to compare the Lie-basis result against dense Schrödinger evolution on a small system.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np
import torch

from quantum_simulation.pauli_algebra import (
    PauliString,
    PauliSum,
    PauliHamiltonian,
    SparseKet,
    lie_closure_basis_symbolic,
    coefficients_in_lie_closure_basis,
    adjoint_generator_in_lie_closure_basis,
    evolve_operator_in_lie_closure_basis,
    basis_expectations_for_sparse_ket,
    basis_expectations_for_density_matrix,
    expectation_value_in_lie_closure_basis,
    heisenberg_expectation_in_lie_closure_basis,
    heisenberg_expectation_for_density_matrix_in_lie_closure_basis,
)

np.set_printoptions(precision=4, suppress=True)
device = "cpu"


## Build a small closed problem

For clarity, start with two qubits. The Hamiltonian couples the qubits through `XX` and adds local `Z` fields. The observable is `Z` on site 0.


In [3]:
hamiltonian = PauliSum([
    PauliString("XX", 0.7),
    PauliString("ZI", -0.3),
    PauliString("IZ", -0.2),
]).simplify()
observable = PauliString("ZI")

print("H =", hamiltonian)
print("O =", observable)


H = ((0.7+0j))*XX + ((-0.3+0j))*ZI + ((-0.2+0j))*IZ
O = ZI


## Step 1: Lie-closure basis

The basis is generated by repeatedly adding commutators and orthonormalizing under the symbolic Frobenius inner product.


In [4]:
basis = lie_closure_basis_symbolic(
    [hamiltonian, PauliSum([observable])],
    atol=1e-10,
    rtol=1e-10,
    max_iter=50,
)

print("basis dimension:", len(basis))
for i, q in enumerate(basis):
    print(f"Q[{i}] = {q}")

assert len(basis) > 0
for i, qi in enumerate(basis):
    for j, qj in enumerate(basis):
        expected = 1.0 if i == j else 0.0
        assert np.allclose(qi.frob_inner(qj), expected, atol=1e-8)


basis dimension: 6
Q[0] = ((0.4445004445006667+0j))*XX + ((-0.19050019050028574+0j))*ZI + ((-0.1270001270001905+0j))*IZ
Q[1] = ((0.46228744025698437+0j))*ZI + ((0.1831704951961636+0j))*XX + ((-0.05233442719890389+0j))*IZ
Q[2] = (-0.49999999999999994j)*YX
Q[3] = ((0.05233442719890372+0j))*XX + ((-0.46228744025698426+0j))*YY + ((0.18317049519616363+0j))*IZ
Q[4] = ((-0.12700012700019056+0j))*XX + ((-0.1905001905002858+0j))*YY + ((-0.4445004445006668+0j))*IZ
Q[5] = (-0.49999999999999994j)*XY


## Step 2: Project the observable and build the adjoint matrix

`coefficients_in_lie_closure_basis` gives the initial observable coordinates. `adjoint_generator_in_lie_closure_basis` represents commutation with the Hamiltonian.


In [5]:
w0 = coefficients_in_lie_closure_basis(basis, observable, device=device)
M = adjoint_generator_in_lie_closure_basis(basis, hamiltonian, device=device)

print("w0 =", w0.cpu().numpy())
print("M shape =", tuple(M.shape))
assert w0.shape == (len(basis),)
assert M.shape == (len(basis), len(basis))


w0 = [-0.762 +0.j  1.8491+0.j  0.    +0.j  0.    +0.j  0.    +0.j  0.    +0.j]
M shape = (6, 6)


## Step 3: Evolve the observable coefficients

The helper applies `exp((i t / hbar) ad_H)` in basis-coordinate form.


In [6]:
time = 0.4
w_t, M_scaled, w_0_again = evolve_operator_in_lie_closure_basis(
    basis,
    hamiltonian,
    observable,
    time=time,
    hbar=1.0,
    device=device,
)

print("w_t =", w_t.cpu().numpy())
print("scaled M shape =", tuple(M_scaled.shape))
assert torch.allclose(w0, w_0_again)


w_t = [-0.762 +0.j      1.521 +0.j      0.    +1.0474j -0.0922+0.j
 -0.0016+0.j      0.    -0.0138j]
scaled M shape = (6, 6)


## Step 4: Contract with state moments

For a sparse basis state, compute all basis moments and contract them with the evolved coefficients.


In [7]:
state = SparseKet.basis(n=2, index=0b00, device=device)
mu = basis_expectations_for_sparse_ket(basis, state, device=device)
value_manual = expectation_value_in_lie_closure_basis(w_t, mu)
value_helper, w_t_helper, M_helper, mu_helper = heisenberg_expectation_in_lie_closure_basis(
    basis,
    hamiltonian,
    observable,
    state,
    time=time,
    device=device,
)

print("mu =", mu.cpu().numpy())
print("manual expectation =", value_manual)
print("helper expectation =", value_helper)

assert np.allclose(value_manual, value_helper)
assert torch.allclose(w_t, w_t_helper)
assert torch.allclose(mu, mu_helper)


mu = [-0.3175+0.j  0.41  +0.j  0.    +0.j  0.1832+0.j -0.4445+0.j  0.    +0.j]
manual expectation = (0.8492915014244807+0j)
helper expectation = (0.8492915014244807+0j)


## Dense cross-check

For a two-qubit example, dense evolution is cheap. This confirms the Lie-basis Heisenberg result against Schrödinger-picture state evolution.


In [8]:
H_dense = torch.as_tensor(hamiltonian.to_matrix(), dtype=torch.complex128)
O_dense = torch.as_tensor(observable.to_matrix(), dtype=torch.complex128)
psi0 = torch.zeros(4, dtype=torch.complex128)
psi0[0b00] = 1.0

U = torch.matrix_exp(-1j * time * H_dense)
psi_t = U @ psi0
value_dense = (psi_t.conj() @ (O_dense @ psi_t)).item()

print("Lie-basis value =", value_helper)
print("dense value     =", value_dense)
print("absolute error  =", abs(value_helper - value_dense))
assert np.allclose(value_helper, value_dense, atol=1e-10)


Lie-basis value = (0.8492915014244807+0j)
dense value     = (0.8492915014244807+0j)
absolute error  = 0.0


## Density-matrix version

The same workflow works for a dense mixed state through `heisenberg_expectation_for_density_matrix_in_lie_closure_basis`.


In [9]:
rho = np.zeros((4, 4), dtype=complex)
rho[0b00, 0b00] = 0.65
rho[0b11, 0b11] = 0.35

value_rho, w_t_rho, M_rho, mu_rho = heisenberg_expectation_for_density_matrix_in_lie_closure_basis(
    basis,
    hamiltonian,
    observable,
    rho,
    time=time,
    device=device,
)
mu_rho_manual = basis_expectations_for_density_matrix(basis, rho, device=device)

print("density-matrix expectation =", value_rho)
print("rho moments =", mu_rho.cpu().numpy())
assert torch.allclose(mu_rho, mu_rho_manual)
assert torch.allclose(w_t_rho, w_t)


density-matrix expectation = (0.25478745042734424+0j)
rho moments = [-0.0953+0.j  0.123 +0.j  0.    +0.j  0.055 +0.j -0.1334+0.j  0.    +0.j]


## Builder-integrated workflow

If the Hamiltonian came from a `PauliHamiltonian` or model builder, the same steps are available as methods.


In [10]:
builder = PauliHamiltonian(lattice_length=2)
builder.add_pauli_string("XX", 0.7)
builder.add_pauli_string("ZI", -0.3)
builder.add_pauli_string("IZ", -0.2)

basis_from_builder = builder.lie_closure_basis([observable], atol=1e-10, rtol=1e-10)
value_builder, _, _, _ = builder.expectation(
    basis_from_builder,
    observable,
    state,
    time=time,
    device=device,
)

print("builder basis dimension:", len(basis_from_builder))
print("builder expectation:", value_builder)
assert np.allclose(value_builder, value_dense, atol=1e-10)


builder basis dimension: 6
builder expectation: (0.8492915014244807+0j)
